In [1]:
from sionna.rt import load_scene

scene = load_scene(
    "3D_scene_sionna_with_VR_2people.xml",
    merge_shapes=False
)

print(scene.objects)

scene.preview()

2026-08-07 07:16:11 WARN  [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).
{'elm__4': <sionna.rt.scene_object.SceneObject object at 0xe783d66cb380>, 'elm__6': <sionna.rt.scene_object.SceneObject object at 0xe783d654ed50>, 'elm__7': <sionna.rt.scene_object.SceneObject object at 0xe783d654ee90>, 'elm__9': <sionna.rt.scene_object.SceneObject object at 0xe783d645ce90>, 'elm__10': <sionna.rt.scene_object.SceneObject object at 0xe783d645cfc0>, 'elm__13': <sionna.rt.scene_object.SceneObject object at 0xe783d6559a30>, 'elm__14': <sionna.rt.scene_object.SceneObject object at 0xe783d63d5bf0>, 'elm__15': <sionna.rt.scene_object.SceneObject object at 0xe783d63d5d00>, 'elm__16': <sionna.rt.scene_object.SceneObject object at 0xe783d6478c50>, 'elm__17': <sionna.rt.scene_object.SceneObject object at 0xe783d6478d50>, 'elm__18': <sionna.rt.scene_object.SceneObject object at 0xe783d63fac60>, 'elm__19': <sionna.rt.scene_object.SceneObject object at 0xe783d63fa9

In [2]:

# UE2 objects and initial reference positions
human2_body = scene.get("human2_body")
human2_hair = scene.get("human2_hair")

HUMAN2_BODY = human2_body.position.numpy().reshape(-1).astype(float)
HUMAN2_HAIR = human2_hair.position.numpy().reshape(-1).astype(float)
HUMAN2_HAIR_OFFSET = HUMAN2_HAIR - HUMAN2_BODY

print("human2 body start:", HUMAN2_BODY)
print("human2 hair start:", HUMAN2_HAIR)


human2 body start: [2.00010204 2.15150166 0.88661909]


In [3]:
from sionna.rt import PlanarArray
from sionna.rt import (

    Transmitter,
    Receiver
)

scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

tx = Transmitter(
    name="tx",
    position=[0, 3, 1.5],
    display_radius=0.05
)

rx = Receiver(
    name="rx",
    position=[0, -3, 1.5],
    display_radius=0.05
)

scene.add(tx)
scene.add(rx)
tx.look_at(rx)

print("Tx/Rx added.")
print(scene.transmitters)
print(scene.receivers)

Tx/Rx added.
{'tx': <sionna.rt.radio_devices.transmitter.Transmitter object at 0xe783d6482f90>}
{'rx': <sionna.rt.radio_devices.receiver.Receiver object at 0xe783d64830e0>}


In [ ]:

from sionna.rt import PathSolver
import numpy as np
import os
from sionna.rt import Camera

solver = PathSolver()

os.makedirs("frames", exist_ok=True)
cam = Camera(
    position=[4.0, -6.0, 4.0],
    look_at=[1.25, 0.0, 1.2]
)

# ============================================================
# Objects
# ============================================================
rx = scene.get("rx")
human_body = scene.get("elm__13")
headset = scene.get("headset_rx")
human2_body = scene.get("human2_body")
human2_hair = scene.get("human2_hair")

# IMPORTANT: Run this cell from a freshly loaded scene so these
# reference positions are not contaminated by a previous episode.
BODY = human_body.position.numpy().reshape(-1).astype(float)
HEADSET = headset.position.numpy().reshape(-1).astype(float)
HUMAN2_BODY = human2_body.position.numpy().reshape(-1).astype(float)
HUMAN2_HAIR = human2_hair.position.numpy().reshape(-1).astype(float)

HEADSET_OFFSET = HEADSET - BODY
HUMAN2_HAIR_OFFSET = HUMAN2_HAIR - HUMAN2_BODY

print("UE1 body start :", BODY)
print("UE1 headset    :", HEADSET)
print("UE2 body start :", HUMAN2_BODY)

# ============================================================
# Simulation parameters
# ============================================================
dt = 0.1                     # 100 ms / slot
total_duration = 4.0         # total episode length
num_slots = int(total_duration / dt) + 1

# First-version randomness: only speed + turn time
RANDOM_SEED = None           # set an integer (e.g. 42) to reproduce an episode
rng = np.random.default_rng(RANDOM_SEED)

UE1_SPEED_RANGE = (0.7, 1.3)   # m/s
UE2_SPEED_RANGE = (0.5, 1.1)   # m/s

ue1_speed = float(rng.uniform(*UE1_SPEED_RANGE))
ue2_speed = float(rng.uniform(*UE2_SPEED_RANGE))

# Both UEs start by turning 90 degrees to face the +X walking direction.
START_HEADING_DEG = 90.0
ue1_heading = float(np.deg2rad(START_HEADING_DEG))
ue2_heading = float(np.deg2rad(START_HEADING_DEG))

# U-turn: 180 degrees over 12 slots = 1.2 s, -15 degrees / slot.
TURN_SLOTS = 12
TURN_STEP_DEG = -15.0

# ============================================================
# 4 m x 4 m movement area
# Current scene coordinates are offset, so use a 4 m x 4 m box
# that contains both starting positions.
# ============================================================
ROOM_X_MIN = 0.0
ROOM_X_MAX = 4.0
ROOM_Y_MIN = -0.5
ROOM_Y_MAX = 3.5
BOUNDARY_MARGIN = 0.05

# Keep motion along X, so the Y coordinates stay fixed.
assert ROOM_Y_MIN <= BODY[1] <= ROOM_Y_MAX, "UE1 starts outside room Y bounds"
assert ROOM_Y_MIN <= HUMAN2_BODY[1] <= ROOM_Y_MAX, "UE2 starts outside room Y bounds"

# ============================================================
# Helpers
# ============================================================
def rotate_y(vec, angle):
    """Rotation convention already verified for this imported scene."""
    c = np.cos(angle)
    s = np.sin(angle)
    R = np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])
    return R @ np.asarray(vec, dtype=float)


def sample_safe_turn_slot(start_x, speed, rng, earliest_s=0.7):
    """
    Random turn time, but constrained so the UE cannot hit x=ROOM_X_MAX
    before the U-turn starts.
    """
    max_forward_time = (
        ROOM_X_MAX - BOUNDARY_MARGIN - float(start_x)
    ) / speed

    # Need enough time to complete the U-turn within this episode.
    latest_by_episode = total_duration - TURN_SLOTS * dt - 0.2
    latest_s = min(max_forward_time, latest_by_episode)

    # If geometry is tight, fall back to the latest feasible slot.
    earliest_s = min(earliest_s, latest_s)
    if latest_s <= 0:
        raise ValueError("No valid forward walking interval inside the 4x4 room.")

    turn_time = float(rng.uniform(max(0.1, earliest_s), max(0.1, latest_s)))
    return int(round(turn_time / dt))


ue1_turn_slot = sample_safe_turn_slot(BODY[0], ue1_speed, rng)
ue2_turn_slot = sample_safe_turn_slot(HUMAN2_BODY[0], ue2_speed, rng)

ue1_turn_end = ue1_turn_slot + TURN_SLOTS
ue2_turn_end = ue2_turn_slot + TURN_SLOTS

print(f"UE1 speed      = {ue1_speed:.3f} m/s")
print(f"UE1 turn start = slot {ue1_turn_slot} ({ue1_turn_slot*dt:.1f} s)")
print(f"UE2 speed      = {ue2_speed:.3f} m/s")
print(f"UE2 turn start = slot {ue2_turn_slot} ({ue2_turn_slot*dt:.1f} s)")

# ============================================================
# Rx offset on headset
# ============================================================
RX_LOCAL_OFFSET = np.array([0.0, -0.06, 0.0])

records = []

# Dynamic X positions. Start from the XML positions.
ue1_x = float(BODY[0])
ue2_x = float(HUMAN2_BODY[0])

# ============================================================
# Main episode loop
# ============================================================
for t in range(num_slots):
    current_time = t * dt

    # --------------------------------------------------------
    # UE1 state
    # --------------------------------------------------------
    if t < ue1_turn_slot:
        ue1_phase = "walking_forward"
        if t > 0:
            ue1_x += ue1_speed * dt

    elif t < ue1_turn_end:
        ue1_phase = "turning"
        ue1_heading = (
            ue1_heading + np.deg2rad(TURN_STEP_DEG)
        ) % (2 * np.pi)

    else:
        ue1_phase = "walking_back"
        ue1_x -= ue1_speed * dt

    # Numerical safety: never leave the 4x4 room.
    ue1_x = float(np.clip(
        ue1_x,
        ROOM_X_MIN + BOUNDARY_MARGIN,
        ROOM_X_MAX - BOUNDARY_MARGIN
    ))

    # --------------------------------------------------------
    # UE2 state
    # UE2 ALSO starts at 90 degrees and performs its own U-turn.
    # --------------------------------------------------------
    if t < ue2_turn_slot:
        ue2_phase = "walking_forward"
        if t > 0:
            ue2_x += ue2_speed * dt

    elif t < ue2_turn_end:
        ue2_phase = "turning"
        ue2_heading = (
            ue2_heading + np.deg2rad(TURN_STEP_DEG)
        ) % (2 * np.pi)

    else:
        ue2_phase = "walking_back"
        ue2_x -= ue2_speed * dt

    ue2_x = float(np.clip(
        ue2_x,
        ROOM_X_MIN + BOUNDARY_MARGIN,
        ROOM_X_MAX - BOUNDARY_MARGIN
    ))

    # --------------------------------------------------------
    # UE1 geometry
    # --------------------------------------------------------
    body_pos = np.array([
        ue1_x,
        float(BODY[1]),
        float(BODY[2])
    ])

    ue1_orientation = [float(ue1_heading), 0.0, 0.0]
    human_body.orientation = ue1_orientation
    headset.orientation = ue1_orientation
    human_body.position = body_pos.tolist()

    # Keep the headset at the face while the body turns.
    # Translation offset is fixed; only the headset itself rotates.
    headset_pos = body_pos + HEADSET_OFFSET
    headset.position = headset_pos.tolist()

    # Rx follows the headset face direction.
    rx_offset_rotated = rotate_y(RX_LOCAL_OFFSET, ue1_heading)
    rx_pos = headset_pos + rx_offset_rotated
    rx.position = rx_pos.tolist()

    # --------------------------------------------------------
    # UE2 geometry
    # --------------------------------------------------------
    human2_pos = np.array([
        ue2_x,
        float(HUMAN2_BODY[1]),
        float(HUMAN2_BODY[2])
    ])

    ue2_orientation = [float(ue2_heading), 0.0, 0.0]
    human2_body.orientation = ue2_orientation
    human2_hair.orientation = ue2_orientation
    human2_body.position = human2_pos.tolist()

    # Keep UE2 hair attached to the body reference point.
    human2_hair.position = (
        human2_pos + HUMAN2_HAIR_OFFSET
    ).tolist()

    # --------------------------------------------------------
    # Safety check: both UEs remain inside 4x4 movement area
    # --------------------------------------------------------
    assert ROOM_X_MIN <= body_pos[0] <= ROOM_X_MAX
    assert ROOM_Y_MIN <= body_pos[1] <= ROOM_Y_MAX
    assert ROOM_X_MIN <= human2_pos[0] <= ROOM_X_MAX
    assert ROOM_Y_MIN <= human2_pos[1] <= ROOM_Y_MAX

    # --------------------------------------------------------
    # Ray tracing
    # --------------------------------------------------------
    paths = solver(
        scene=scene,
        max_depth=3,
        los=True,
        specular_reflection=True,
        diffuse_reflection=False,
        refraction=False,
        synthetic_array=True,
        seed=1
    )

    # Overall phase retained for your existing plotting code.
    overall_phase = "turning" if (
        ue1_phase == "turning" or ue2_phase == "turning"
    ) else "walking"

    records.append({
        "slot": t,
        "time_s": current_time,

        # UE1 / headset / Rx
        "x": float(headset_pos[0]),
        "y": float(headset_pos[1]),
        "z": float(headset_pos[2]),
        "heading_rad": float(ue1_heading),
        "heading_deg": float(np.rad2deg(ue1_heading)),
        "ue1_speed_mps": float(ue1_speed),
        "ue1_turn_slot": int(ue1_turn_slot),
        "ue1_phase": ue1_phase,
        "rx_x": float(rx_pos[0]),
        "rx_y": float(rx_pos[1]),
        "rx_z": float(rx_pos[2]),

        # UE2
        "human2_x": float(human2_pos[0]),
        "human2_y": float(human2_pos[1]),
        "human2_z": float(human2_pos[2]),
        "human2_heading_rad": float(ue2_heading),
        "human2_heading_deg": float(np.rad2deg(ue2_heading)),
        "human2_speed_mps": float(ue2_speed),
        "human2_turn_slot": int(ue2_turn_slot),
        "human2_phase": ue2_phase,

        "phase": overall_phase,
        "paths": paths
    })

    print(
        f"slot={t:02d} time={current_time:.1f}s | "
        f"UE1 x={ue1_x:.2f} h={np.rad2deg(ue1_heading):.0f}° {ue1_phase} | "
        f"UE2 x={ue2_x:.2f} h={np.rad2deg(ue2_heading):.0f}° {ue2_phase}",
        flush=True
    )

print("Total records:", len(records))

scene.preview(
    paths=paths,
    show_devices=True
)


Old single-turn test disabled. UE1 and UE2 turning are now handled inside the main episode loop above.


Old UE1-only U-turn loop disabled. Both UEs now turn and keep moving within the 4x4 bounds in the main loop.


In [ ]:
print(records[0])
print(records[1])
print(len(records))

In [ ]:
import numpy as np

num_subcarriers = 64
subcarrier_spacing = 30e3 

frequencies = (
    np.arange(num_subcarriers)
    - num_subcarriers // 2
) * subcarrier_spacing

for record in records:

    paths = record["paths"]

    csi = paths.cfr(
        frequencies=frequencies,
        normalize_delays=False,
        normalize=False,
        out_type="numpy"
    )

    # SISO: remove dimensions of size 1
    csi = np.squeeze(csi)

    record["csi"] = csi

    print(
        f"slot={record['slot']}, "
        f"time={record['time_s']:.1f}s, "
        f"CSI shape={csi.shape}"
    )

In [ ]:
# =========================
# Throughput parameters
# =========================

P_tx_dBm = 20.0       # 100 mW
noise_dBm = -90.0     # total equivalent noise power
subcarrier_spacing = 30e3

P_tx = 10 ** ((P_tx_dBm - 30) / 10)
noise_power = 10 ** ((noise_dBm - 30) / 10)

In [ ]:
for record in records:

    csi = record["csi"]

    # 每個 subcarrier 的 channel gain
    channel_gain = np.abs(csi) ** 2

    # 每個 subcarrier 的 SNR
    snr = P_tx * channel_gain / noise_power

    # Shannon throughput
    throughput_bps = np.sum(
        subcarrier_spacing * np.log2(1 + snr)
    )

    record["snr"] = snr
    record["throughput_bps"] = float(throughput_bps)
    record["throughput_mbps"] = float(throughput_bps / 1e6)

    print(
        f"slot={record['slot']}, "
        f"time={record['time_s']:.1f}s, "
        f"throughput={record['throughput_mbps']:.2f} Mbps"
    )

In [ ]:
import matplotlib.pyplot as plt

times = np.array([
    r["time_s"]
    for r in records
])

throughputs = np.array([
    r["throughput_mbps"]
    for r in records
])

phases = [
    r["phase"]
    for r in records
]

plt.figure(figsize=(12, 5))

plt.plot(
    times,
    throughputs,
    marker="o"
)

# 找 turning 開始的時間
turn_indices = [
    i for i, phase in enumerate(phases)
    if phase == "turning"
]

if len(turn_indices) > 0:

    turn_start = times[turn_indices[0]]

    plt.axvline(
        turn_start,
        linestyle="--"
    )

    plt.text(
        turn_start,
        max(throughputs),
        "Start turning",
        rotation=90,
        va="top"
    )

plt.xlabel("Time (s)")
plt.ylabel("Throughput (Mbps)")
plt.title("Throughput During UE Movement")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:

import numpy as np

# ==========================================
# records -> numpy dataset
# ==========================================
slots = np.array([r["slot"] for r in records], dtype=np.int32)
time_s = np.array([r["time_s"] for r in records], dtype=np.float32)

position = np.array([
    [r["x"], r["y"], r["z"]]
    for r in records
], dtype=np.float32)

rx_position = np.array([
    [r["rx_x"], r["rx_y"], r["rx_z"]]
    for r in records
], dtype=np.float32)

heading_rad = np.array([r["heading_rad"] for r in records], dtype=np.float32)
heading_deg = np.array([r["heading_deg"] for r in records], dtype=np.float32)

human2_position = np.array([
    [r["human2_x"], r["human2_y"], r["human2_z"]]
    for r in records
], dtype=np.float32)

human2_heading_rad = np.array([
    r["human2_heading_rad"] for r in records
], dtype=np.float32)

human2_heading_deg = np.array([
    r["human2_heading_deg"] for r in records
], dtype=np.float32)

# CSI [T, 64]
csi = np.stack([r["csi"] for r in records])
csi_real = np.real(csi).astype(np.float32)
csi_imag = np.imag(csi).astype(np.float32)

throughput_mbps = np.array([
    r["throughput_mbps"] for r in records
], dtype=np.float32)

phase = np.array([r["phase"] for r in records])
ue1_phase = np.array([r["ue1_phase"] for r in records])
human2_phase = np.array([r["human2_phase"] for r in records])

# Episode-level random parameters
ue1_speed_mps = np.float32(records[0]["ue1_speed_mps"])
human2_speed_mps = np.float32(records[0]["human2_speed_mps"])
ue1_turn_slot = np.int32(records[0]["ue1_turn_slot"])
human2_turn_slot = np.int32(records[0]["human2_turn_slot"])

np.savez(
    "walking_dataset_both_UEs.npz",
    slot=slots,
    time_s=time_s,
    position=position,
    rx_position=rx_position,
    heading_rad=heading_rad,
    heading_deg=heading_deg,
    human2_position=human2_position,
    human2_heading_rad=human2_heading_rad,
    human2_heading_deg=human2_heading_deg,
    csi_real=csi_real,
    csi_imag=csi_imag,
    throughput_mbps=throughput_mbps,
    phase=phase,
    ue1_phase=ue1_phase,
    human2_phase=human2_phase,
    ue1_speed_mps=ue1_speed_mps,
    human2_speed_mps=human2_speed_mps,
    ue1_turn_slot=ue1_turn_slot,
    human2_turn_slot=human2_turn_slot,
)

print("saved: walking_dataset_both_UEs.npz")


In [ ]:
data = np.load(
    "walking_dataset.npz",
    allow_pickle=True
)

print(data.files)
print("position:", data["position"].shape)
print("CSI:", data["csi_real"].shape)
print("throughput:", data["throughput_mbps"].shape)

In [ ]:
r0 = records[0]

print("slot =", r0["slot"])
print("time =", r0["time_s"])
print("CSI =", r0["csi"])
print("max |H| =", np.max(np.abs(r0["csi"])))
print("mean |H| =", np.mean(np.abs(r0["csi"])))
print("throughput =", r0["throughput_mbps"])